# Context-aware toxicity: message ? reply chain ? prediction

This notebook runs the **same Python modules as the command-line pipeline**. There is no topic selector, keyword decision gate, or hard-coded prediction. It creates individual text embeddings, contextual message embeddings, and a whole-conversation embedding. The binary score belongs to the **target message**, not every message in the paragraph/history.

Open this notebook from the project folder or `notebooks/`. Select a Python kernel with the dependencies installed. Run `python -m pip install -r requirements-contextual.txt` once in that environment. Initial data/model download requires internet; after downloading and training, inference uses the saved local encoder and tokenizer.

**Run All performs real training**, potentially taking significant time. Missing network/data produces an actionable error, never synthetic replacement data or fake scores. Configuration is in `configs/contextual.json`.

In [1]:
import os
import sys
from pathlib import Path

# Discover the repository without a machine-specific drive or cwd assumption.
start = Path(os.environ.get("TOXICITY_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "configs" / "contextual.json").is_file()
             and (p / "contextual" / "config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Launch Jupyter from the project folder, or set TOXICITY_PROJECT_ROOT to its location.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from contextual.config import load_config, location
cfg = load_config()  # TOXICITY_CONFIG may select another JSON file.
print("Project:", ROOT)
print("Data:", location(cfg, "prepared"))
print("Checkpoint:", location(cfg, "checkpoint"))

# Reload modules after editing Python files in an existing notebook session.
import importlib
import contextual.data
importlib.reload(contextual.data)


Project: D:\NLP_projects\Context_Aware_Toxicity_classifier
Data: D:\NLP_projects\Context_Aware_Toxicity_classifier\data\processed\contextual.jsonl
Checkpoint: D:\NLP_projects\Context_Aware_Toxicity_classifier\models\checkpoints\contextual


<module 'contextual.data' from 'D:\\NLP_projects\\Context_Aware_Toxicity_classifier\\contextual\\data.py'>

## 1. Download authentic parent?child conversations

Cornell's CGA-WIKI supplies reply IDs and **comment-level personal-attack labels**. Section headers are excluded. The corpus-level future-attack annotation is never used as the current target label. The archive and SHA-256 provenance stay in the configured raw directory.

Existing Telugu and Tamil code-mixed parquet files are optional supplemental **single-message** examples. They do not supply missing context labels. Custom threads can be supplied using the documented JSONL schema. No automatic translation or template-generated labels are treated as ground truth.

In [2]:
from contextual.data import download, prepare
raw_directory = download(cfg)
print("Verified and extracted:", raw_directory)

Using verified local archive: D:\NLP_projects\Context_Aware_Toxicity_classifier\data\raw\contextual\cga-wiki.zip
Verified and extracted: D:\NLP_projects\Context_Aware_Toxicity_classifier\data\raw\contextual\cga-wiki


## Prepare and validate downloaded data
Rerun this cell independently. Invalid ancestry is excluded and counted; no parent links are invented.

In [3]:
manifest = prepare(cfg)
manifest

{'examples': 78839,
 'real_context_examples': 14598,
 'cga_skipped_missing_ancestry': 2509,
 'cga_skipped_empty_ancestry': 543,
 'cga_skipped_invalid_ancestry': 177,
 'splits': {'train': 54960, 'val': 11893, 'test': 11986},
 'sources': [{'source': 'cultural_safety_telugu',
   'status': 'loaded',
   'path': 'D:\\NLP_projects\\Context_Aware_Toxicity_classifier\\data\\processed\\cultural_safety_telugu.parquet'},
  {'source': 'dravidian_codemix_tamil',
   'status': 'loaded',
   'path': 'D:\\NLP_projects\\Context_Aware_Toxicity_classifier\\data\\processed\\dravidian_codemix.parquet'}],
 'source_counts': {'cga_wiki': 22959,
  'cultural_safety_telugu': 11961,
  'dravidian_codemix_tamil': 43919},
 'split_policy': 'New grouped splits; all shared message text and CGA articles stay in one split. Not the original benchmark splits.',
 'task': 'Binary target toxicity; CGA personal attacks and local toxicity labels are related but not identical constructs.'}

## 2. Inspect data and leakage-safe splits

Each sample follows the real reply path, oldest ancestor to target. The most recent configured number of turns is retained. Whole conversations, CGA articles, and any groups sharing message text stay in one split. These are new project splits, not the published benchmark splits. Both classes must occur in each split.

In [4]:
from collections import Counter
from contextual.data import read_jsonl
examples = read_jsonl(location(cfg, "prepared"))
print("Sources:", Counter(e["source"] for e in examples))
print("Splits:", Counter(e["split"] for e in examples))
print("With ancestry:", sum(len(e["texts"]) > 1 for e in examples))
assert manifest["real_context_examples"] > 0
example = next(e for e in examples if len(e["texts"]) > 1)
{"source": example["source"], "turn_count": len(example["texts"]), "label": example["label"]}

Sources: Counter({'dravidian_codemix_tamil': 43919, 'cga_wiki': 22959, 'cultural_safety_telugu': 11961})
Splits: Counter({'train': 54960, 'test': 11986, 'val': 11893})
With ancestry: 14598


{'source': 'cga_wiki', 'turn_count': 2, 'label': 0}

## 3. Train the hierarchical Transformer

A frozen IndicBERT encoder embeds each message independently. Long messages are split into token chunks and pooled by content-token count so the tail is retained. Embedding width comes from the model configuration. A trainable Transformer combines the ancestor vectors with learned positions; learned attention pools the whole reply path. The classifier combines the target alone, the contextual target, and the conversation vector.

The pretrained encoder is frozen for reproducible cached embeddings and modest GPU usage. It is **not** a supervised sentence-similarity model. The conversation Transformer and classifier learn from the supplied labelled data. Early stopping selects validation average precision; the binary decision threshold is fitted on validation F1. No test label is used for training or tuning.

## Load pretrained encoder separately
This is transfer learning: IndicBERT supplies pretrained representations and the conversation classifier learns from labelled threads. The first run downloads model weights. UNEXPECTED keys named cls.* are unused pretraining heads; they are expected when loading the base encoder. Other loading errors must be investigated.

In [5]:
from contextual.embeddings import TextEncoder, cached_embeddings
encoder = TextEncoder(cfg)
print("Encoder ready:", encoder.device, "dimensions:", encoder.dimension)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-Sam-TLM
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder ready: cuda dimensions: 768


## Create and cache embeddings
This is the slowest initial stage. Completed embeddings are reused when the data and encoder settings match. Keep the kernel running until the cache is saved.

In [6]:
vectors, lookup = cached_embeddings(cfg, examples, encoder)
print("Cached vectors:", tuple(vectors.shape))
del encoder, vectors, lookup
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Embedding progress: 685/78,320 texts
Embedding progress: 1,383/78,320 texts
Embedding progress: 2,088/78,320 texts
Embedding progress: 2,784/78,320 texts
Embedding progress: 3,451/78,320 texts
Embedding progress: 4,100/78,320 texts
Embedding progress: 4,668/78,320 texts
Embedding progress: 5,234/78,320 texts
Embedding progress: 5,612/78,320 texts
Embedding progress: 6,187/78,320 texts
Embedding progress: 6,732/78,320 texts
Embedding progress: 7,281/78,320 texts
Embedding progress: 7,846/78,320 texts
Embedding progress: 8,432/78,320 texts
Embedding progress: 8,975/78,320 texts
Embedding progress: 9,515/78,320 texts
Embedding progress: 10,046/78,320 texts
Embedding progress: 10,665/78,320 texts
Embedding progress: 11,196/78,320 texts
Embedding progress: 11,793/78,320 texts
Embedding progress: 12,301/78,320 texts
Embedding progress: 12,884/78,320 texts
Embedding progress: 13,462/78,320 texts
Embedding progress: 14,047/78,320 texts
Embedding progress: 14,616/78,320 texts
Embedding progress

In [7]:
from contextual.training import train
report = train(cfg)  # Includes held-out evaluation and saves encoder/tokenizer/head/metadata.
report

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-Sam-TLM
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 78839 examples with ai4bharat/IndicBERTv2-MLM-Sam-TLM
Reusing verified embedding cache
{'epoch': 1, 'loss': 0.743571812566695, 'validation_ap': 0.654853225606483}
{'epoch': 2, 'loss': 0.6749909723110332, 'validation_ap': 0.6859912327977082}
{'epoch': 3, 'loss': 0.6553537213830093, 'validation_ap': 0.6993678530485882}
{'epoch': 4, 'loss': 0.6410669465489659, 'validation_ap': 0.6986063659807193}
{'epoch': 5, 'loss': 0.6265454963630237, 'validation_ap': 0.7203213093123424}
{'epoch': 6, 'loss': 0.6198087329428743, 'validation_ap': 0.7235350202389145}
{'epoch': 7, 'loss': 0.6123931613318162, 'validation_ap': 0.7234038275729733}
{'epoch': 8, 'loss': 0.606019210100729, 'validation_ap': 0.7209029897602967}
{'epoch': 9, 'loss': 0.6029972192572215, 'validation_ap': 0.7247027754798986}
{'epoch': 10, 'loss': 0.5960382812583821, 'validation_ap': 0.7278963368796293}
{'epoch': 11, 'loss': 0.5962698667790476, 'validation_ap': 0.7251704964040737}
{'epoch': 12, 'loss': 0.5917849901183243, 'val

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Reusing verified embedding cache


{'threshold': 0.6197893619537354,
 'test': {'n': 11986,
  'positive_count': 2669,
  'f1': 0.6584524396254312,
  'roc_auc': 0.8938936842305486,
  'average_precision': 0.7112610612668083,
  'confusion_matrix': [[7903, 1414], [665, 2004]]},
 'context_removed_ablation': {'n': 11986,
  'positive_count': 2669,
  'f1': 0.6586345381526104,
  'roc_auc': 0.8948107603978966,
  'average_precision': 0.7160622578528646,
  'confusion_matrix': [[7978, 1339], [701, 1968]]},
 'mean_absolute_probability_change': 0.01207024611562644,
 'has_context_training': True,
 'by_source': {'cga_wiki': {'n': 3612,
   'positive_count': 274,
   'f1': 0.6335766423357664,
   'roc_auc': 0.9476193183557617,
   'average_precision': 0.7008004896493869,
   'confusion_matrix': [[3144, 194], [57, 217]]},
  'cultural_safety_telugu': {'n': 1787,
   'positive_count': 569,
   'f1': 0.8248673237300985,
   'roc_auc': 0.9553894280577513,
   'average_precision': 0.8937635043338039,
   'confusion_matrix': [[1012, 206], [25, 544]]},
  'd

## 4. Read measured held-out results

Compare full context with the same trained model given only the target. This ablation is not a separately trained baseline. Report per-source metrics because personal attack, toxicity, and offensiveness differ. A multilingual backbone alone does not establish Telugu or Romanized Telugu accuracy. No calibration claim is made for the sigmoid score.

In [8]:
import json
report = json.loads(location(cfg, "report").read_text(encoding="utf-8"))
print(json.dumps(report, indent=2))

{
  "threshold": 0.6197893619537354,
  "test": {
    "n": 11986,
    "positive_count": 2669,
    "f1": 0.6584524396254312,
    "roc_auc": 0.8938936842305486,
    "average_precision": 0.7112610612668083,
    "confusion_matrix": [
      [
        7903,
        1414
      ],
      [
        665,
        2004
      ]
    ]
  },
  "context_removed_ablation": {
    "n": 11986,
    "positive_count": 2669,
    "f1": 0.6586345381526104,
    "roc_auc": 0.8948107603978966,
    "average_precision": 0.7160622578528646,
    "confusion_matrix": [
      [
        7978,
        1339
      ],
      [
        701,
        1968
      ]
    ]
  },
  "mean_absolute_probability_change": 0.01207024611562644,
  "has_context_training": true,
  "by_source": {
    "cga_wiki": {
      "n": 3612,
      "positive_count": 274,
      "f1": 0.6335766423357664,
      "roc_auc": 0.9476193183557617,
      "average_precision": 0.7008004896493869,
      "confusion_matrix": [
        [
          3144,
          194
        ]

## 5. Inspect individual and whole-conversation embeddings

This example uses an actual held-out thread. Replace `target` and `parents` with your text. A paragraph can be passed as a single message. Do not invent parent links between unrelated sentences. The model receives no gaming/politics/domain label.

In [9]:
import numpy as np
from contextual.inference import Predictor
predictor = Predictor(location(cfg, "checkpoint"))
sample = next((e for e in examples if e["split"] == "test" and len(e["texts"]) > 1),
              next(e for e in examples if e["split"] == "test"))
target, parents = sample["texts"][-1], sample["texts"][:-1]
result = predictor.predict(target, history=parents, include_embeddings=True)
print("Individual message vectors:", np.asarray(result["individual_embeddings"]).shape)
print("Contextual message vectors:", np.asarray(result["contextual_embeddings"]).shape)
print("Whole conversation vector:", np.asarray(result["conversation_embedding"]).shape)
{k: v for k, v in result.items() if "embedding" not in k}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Individual message vectors: (2, 768)
Contextual message vectors: (2, 192)
Whole conversation vector: (192,)


{'is_toxic': False,
 'toxicity_score': 0.0006483488832600415,
 'threshold': 0.6197893619537354,
 'isolated_score': 0.0013285453896969557,
 'context_score_delta': -0.0006801965064369142,
 'turn_attention': [0.7344318628311157, 0.2655681073665619],
 'turns_used': 2,
 'ancestors_omitted': 0,
 'has_context_training': True}

## 6. Explicit parent?child inference

For a branching discussion, use `predict_thread`. Only the requested target's actual ancestors are included. Missing parents, duplicate IDs, cycles, and cross-conversation links are rejected. Attention weights describe the model computation; they are not a causal explanation or a measured percentage of intent.

In [10]:
messages = [dict(id=str(i), parent_id=str(i-1) if i else None,
                 conversation_id=sample["group"], text=text)
            for i, text in enumerate(sample["texts"])]
thread_result = predictor.predict_thread(messages, target_id=messages[-1]["id"])
thread_result

{'is_toxic': False,
 'toxicity_score': 0.0006483488832600415,
 'threshold': 0.6197893619537354,
 'isolated_score': 0.0013285453896969557,
 'context_score_delta': -0.0006801965064369142,
 'turn_attention': [0.7344318628311157, 0.2655681073665619],
 'turns_used': 2,
 'ancestors_omitted': 0,
 'has_context_training': True,
 'message_ids': ['0', '1']}

## Files and entry points

- `python run_pipeline.py --all`: download ? prepare ? embed ? train ? evaluate.
- `python run_pipeline.py --predict "Your reply" --parent "Earlier message" --embeddings`.
- `python app/server.py`: local web app; `/docs` exposes the API.
- `python -m pytest tests/test_contextual.py -q`: offline correctness and integration tests.

All paths are repository-relative settings resolved in Python, not relative to the shell's working directory. Local checkpoint inference needs no network. See `README.md` and `docs/CONTEXTUAL_REDESIGN.md` for limitations and custom data format.

Sources: [CGA-WIKI](https://convokit.cornell.edu/documentation/awry.html), [IndicBERT](https://huggingface.co/ai4bharat/IndicBERTv2-MLM-Sam-TLM).